# Step 1A: DDXPlus Clinical Preprocessing & Evidence Decoder Pipeline

## Objectives:
1. **Decode Machine Tokens**: Transform raw codes (`E_48`, `E_54_@_V_161`, `E_56_@_4`) into human-readable clinical concepts.
2. **Symptom vs. Antecedent Distinction**: Explicitly separate active complaints (cough, fever, pain) from patient history and risk factors (allergies, smoking, travel).
3. **Clinical Vignette Generation**: Construct doctor-readable clinical notes for LLM prompts and agent reasoning.
4. **Batch Preprocessing**: Export a structured dataset (`ddxplus_train_10k_decoded.csv`) ready for intake agent dialogue and diagnostic model training.

### 1. Import the Clinical Decoder Module
Importing `DDXPlusDecoder` from our modular `src/` package.

In [1]:
import os
import pandas as pd
import json
from src.decoder import DDXPlusDecoder

decoder = DDXPlusDecoder(
    evidences_path="../data/raw/release_evidences.json",
    conditions_path="../data/raw/release_conditions.json"
)
print("Loaded DDXPlusDecoder successfully!")
print(f"Evidences defined: {len(decoder.evidences)}")
print(f"Conditions defined: {len(decoder.conditions)}")

Loaded DDXPlusDecoder successfully!
Evidences defined: 223
Conditions defined: 49


### 2. Decoding Individual Evidence Tokens
Testing binary evidence (`E_91`), categorical evidence with value code mapping (`E_54_@_V_161`), integer-scale continuous evidence (`E_56_@_4`), and antecedents (`E_48`).

In [2]:
print("--- Binary Evidence (E_91) ---")
print(json.dumps(decoder.decode_token("E_91"), indent=2))

print("\n--- Categorical Evidence (E_54_@_V_161) ---")
print(json.dumps(decoder.decode_token("E_54_@_V_161"), indent=2))

print("\n--- Continuous / Scale Evidence (E_56_@_4) ---")
print(json.dumps(decoder.decode_token("E_56_@_4"), indent=2))

print("\n--- Medical History / Antecedent (E_48) ---")
print(json.dumps(decoder.decode_token("E_48"), indent=2))

--- Binary Evidence (E_91) ---
{
  "token": "E_91",
  "code": "E_91",
  "question_en": "Do you have a fever (either felt or measured with a thermometer)?",
  "is_antecedent": false,
  "data_type": "B",
  "value_code": null,
  "value_meaning": true,
  "clinical_text": "Do you have a fever (either felt or measured with a thermometer)? -> Present"
}

--- Categorical Evidence (E_54_@_V_161) ---
{
  "token": "E_54_@_V_161",
  "code": "E_54",
  "question_en": "Characterize your pain:",
  "is_antecedent": false,
  "data_type": "M",
  "value_code": "V_161",
  "value_meaning": "sensitive",
  "clinical_text": "Characterize your pain: -> sensitive"
}

--- Continuous / Scale Evidence (E_56_@_4) ---
{
  "token": "E_56_@_4",
  "code": "E_56",
  "question_en": "How intense is the pain?",
  "is_antecedent": false,
  "data_type": "C",
  "value_code": "4",
  "value_meaning": "4",
  "clinical_text": "How intense is the pain? -> 4"
}

--- Medical History / Antecedent (E_48) ---
{
  "token": "E_48",
  "cod

### 3. Load Sample Patient and Inspect Raw vs. Decoded Features
Loading Patient 0 from our 10,000-row development split.

In [3]:
df_10k = pd.read_csv("../data/processed/ddxplus_train_10k.csv")
patient_0 = df_10k.iloc[0]
print("Raw Patient Record:")
print(patient_0)

Raw Patient Record:
AGE                                                                      49
DIFFERENTIAL_DIAGNOSIS    [['Allergic sinusitis', 0.4080602882460878], [...
SEX                                                                       M
PATHOLOGY                                                Allergic sinusitis
EVIDENCES                 ['E_86', 'E_87', 'E_124', 'E_169', 'E_181', 'E...
INITIAL_EVIDENCE                                                      E_169


### 4. Decode Full Patient Record
Separating symptoms vs antecedents, decoding the chief complaint (`INITIAL_EVIDENCE`), and fetching ICD-10 and severity for the ground-truth pathology.

In [4]:
decoded_patient_0 = decoder.decode_patient(patient_0)

print(f"Patient Age: {decoded_patient_0['age']} | Sex: {decoded_patient_0['sex']}")
print(f"Ground Truth Pathology: {decoded_patient_0['pathology']['name']} (ICD-10: {decoded_patient_0['pathology']['icd10']}, Severity: {decoded_patient_0['pathology']['severity']}/5)")
print(f"Chief Complaint: {decoded_patient_0['initial_evidence']['clinical_text']}")
print(f"Total Acute Symptoms: {decoded_patient_0['total_symptoms']}")
print(f"Total Antecedents: {decoded_patient_0['total_antecedents']}")

print("\nAcute Symptoms:")
for s in decoded_patient_0['symptoms']:
    print(f"  - {s['clinical_text']}")

print("\nAntecedents / History:")
for a in decoded_patient_0['antecedents']:
    print(f"  - {a['clinical_text']}")

Patient Age: 49 | Sex: M
Ground Truth Pathology: Allergic sinusitis (ICD-10: J30, Severity: 4/5)
Chief Complaint: Is your nose or the back of your throat itchy? -> Present
Total Acute Symptoms: 3
Total Antecedents: 5

Acute Symptoms:
  - Is your nose or the back of your throat itchy? -> Present
  - Do you have nasal congestion or a clear runny nose? -> Present
  - Do you have a cough? -> Present

Antecedents / History:
  - Do you have any close family members who suffer from allergies (any type), hay fever or eczema? -> Present
  - Do you have any family members who have asthma? -> Present
  - Do you have asthma or have you ever had to use a bronchodilator in the past? -> Present
  - Have you traveled out of the country in the last 4 weeks? -> N
  - Do you live in in a big city? -> Present


### 5. Generate Clinical Patient Vignette
This vignette format will be directly used by our LLM diagnostic agent and triage engine.

In [5]:
vignette = decoder.to_clinical_vignette(decoded_patient_0)
print(vignette)

================ CLINICAL PATIENT VIGNETTE ================
DEMOGRAPHICS:
  Age: 49 | Sex: Male

CHIEF COMPLAINT (INITIAL EVIDENCE):
  Is your nose or the back of your throat itchy? -> Present

ACUTE SYMPTOMS (3):
  - Is your nose or the back of your throat itchy? -> Present
  - Do you have nasal congestion or a clear runny nose? -> Present
  - Do you have a cough? -> Present

MEDICAL HISTORY & RISK FACTORS (ANTECEDENTS, 5):
  - Do you have any close family members who suffer from allergies (any type), hay fever or eczema? -> Present
  - Do you have any family members who have asthma? -> Present
  - Do you have asthma or have you ever had to use a bronchodilator in the past? -> Present
  - Have you traveled out of the country in the last 4 weeks? -> N
  - Do you live in in a big city? -> Present

GROUND TRUTH PATHOLOGY:
  Allergic sinusitis (ICD-10: J30, Severity: 4/5)

TOP DIFFERENTIAL DIAGNOSES:
  - Allergic sinusitis (prob: 40.8%, severity: 4)
  - Bronchitis (prob: 34.0%, severity: 

### 6. Batch Decoding & Exporting Processed Dataset
Decoding all 10,000 patients in our development split into structured clinical features.

In [6]:
decoded_df = decoder.decode_dataframe(df_10k)
print("Decoded dataset shape:", decoded_df.shape)

output_path = "../data/processed/ddxplus_train_10k_decoded.csv"
decoded_df.to_csv(output_path, index=False)
print(f"Saved successfully to: {output_path}")
decoded_df.head(3)

Decoded dataset shape: (10000, 11)
Saved successfully to: ../data/processed/ddxplus_train_10k_decoded.csv


,AGE,SEX,PATHOLOGY,PATHOLOGY_ICD10,PATHOLOGY_SEVERITY,INITIAL_EVIDENCE_NAME,INITIAL_EVIDENCE_VALUE,NUM_SYMPTOMS,NUM_ANTECEDENTS,SYMPTOMS_SUMMARY,ANTECEDENTS_SUMMARY
0,49,M,Allergic sinusitis,J30,4,Is your nose or the back of your throat itchy?,True,3,5,Is your nose or the back of your throat itchy? -> Present; Do you have nasal congestion or a clear runny nose? -> Present; Do you have a cough? -> Present,"Do you have any close family members who suffer from allergies (any type), hay fever or eczema? -> Present; Do you have any family members who have asthma? -> Present; Do you have asthma or have you ever had to use a bronchodilator in the past? -> Present; Have you traveled out of the country in the last 4 weeks? -> N; Do you live in in a big city? -> Present"
1,77,F,Myasthenia gravis,G70.0,3,Do you have pain or weakness in your jaw?,True,5,2,"Do you have pain or weakness in your jaw? -> Present; Do you have the perception of seeing two images of a single object seen overlapping or adjacent to each other (double vision)? -> Present; Do you have difficulty articulating words/speaking? -> Present; Do you have difficulty swallowing, or have a feeling of discomfort/blockage when swallowing? -> Present; Do you have a hard time opening/raising one or both eyelids? -> Present",Are there any members of your family who have been diagnosed myasthenia gravis? -> Present; Have you traveled out of the country in the last 4 weeks? -> N
2,2,F,Influenza,j11.1,3,"Do you have any lesions, redness or problems on your skin that you believe are related to the condition you are consulting for?",True,27,1,"Have you had significantly increased sweating? -> Present; Do you have pain somewhere, related to your reason for consulting? -> Present; Characterize your pain: -> sensitive; Characterize your pain: -> heavy; Do you feel pain somewhere? -> side of the neck(R); Do you feel pain somewhere? -> side of the neck(L); Do you feel pain somewhere? -> top of the head; Do you feel pain somewhere? -> forehead; Do you feel pain somewhere? -> occiput; How intense is the pain? -> 2; Does the pain radiate to another location? -> nowhere; How precisely is the pain located? -> 6; How fast did the pain appear? -> 6; Do you feel so tired that you are unable to do your usual activities or are you stuck in your bed all day long? -> Present; Have you had chills or shivers? -> Present; Do you have any lesions, redness or problems on your skin that you believe are related to the condition you are consulting for? -> Present; What color is the rash? -> pink; Do your lesions peel off? -> N; Is the rash swollen? -> 1; Where is the affected region located? -> back of the neck; Where is the affected region located? -> side of the neck(R); Where is the affected region located? -> forehead; How intense is the pain caused by the rash? -> 3; Is the lesion (or are the lesions) larger than 1cm? -> Y; How severe is the itching? -> 4; Do you have nasal congestion or a clear runny nose? -> Present; Do you have a cough? -> Present",Have you traveled out of the country in the last 4 weeks? -> N


## Summary of Step 1A

1. **Clinical Decoder Implemented**: `DDXPlusDecoder` in `src/decoder.py` provides modular, reproducible decoding of evidences and conditions.
2. **Symptom vs. Antecedent Separation**: Clinical features are segregated into acute presenting symptoms vs. chronic medical history and risk factors.
3. **Clinical Vignette Formatter**: Generates human- and LLM-friendly clinical notes with ICD-10 and triage severity scores.
4. **Decoded Dataset Available**: `ddxplus_train_10k_decoded.csv` provides clean, processed features ready for:
   - Multi-turn inquiry simulation (Intake Agent)
   - Differential diagnostic modeling
   - Urgency and triage classification